# 14 — σ(Q) for PDF / Rietveld Pipelines

Downstream area-detector pipelines — radial integration for Rietveld
refinement (GSAS-II, TOPAS), pair-distribution-function (PDF)
analysis (PDFgetX3, diffpy) — consume the **momentum transfer**
`Q = (4π/λ) sin(θ)` per integrated point, not the calibration
parameters themselves.

This notebook walks through the σ(Q) propagation for a typical PDF
workflow:
1. Run the v2 pipeline → MAP geometry + Laplace covariance
2. For each ring's (R, 2θ), compute J_Q = ∂Q/∂(refined params)
3. σ²(Q_k) = J_Q^T · Cov · J_Q
4. Decide if the calibration σ matters for your science

For the headline Varex CeO₂ configuration with only L_sd refined,
σ(Q)/Q ≈ σ(L_sd)/L_sd ≈ 0.78 ppm — **3 orders of magnitude tighter
than typical PDF/Rietveld bin widths**.  Calibration σ is rarely
the limiting source of σ(Q) on a properly-calibrated synchrotron
beamline.


In [1]:
import os, math
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
from pathlib import Path
import time
import numpy as np
import torch
from PIL import Image

from midas_calibrate.params import CalibrationParams as V1Params
from midas_calibrate_v2.compat.from_v1 import spec_from_v1_file
from midas_calibrate_v2.pipelines.single_pv import autocalibrate_pv
from midas_calibrate_v2.loss.pseudo_strain import pseudo_strain_residual
from midas_calibrate_v2.inference.laplace import fisher_at_map

BASE = Path(os.environ.get('V2_TEST_BASE', '/tmp/midas_v2_test'))
PARAMS = BASE / 'refined_MIDAS_params_Ceria_63keV_900mm_100x100_0p5s_aero_0.txt'
IMAGE  = BASE / 'Ceria_63keV_900mm_100x100_0p5s_aero_0_001137.tif'

v1 = V1Params.from_file(PARAMS)
if v1.RBinSize <= 0: v1.RBinSize = 0.25
if v1.EtaBinSize <= 0: v1.EtaBinSize = 5.0
v1.MaxRingRad = max(v1.MaxRingRad, v1.RhoD / max(v1.pxY, 1.0))
v1.Width = max(v1.Width, 800.0)
image = np.array(Image.open(str(IMAGE))).astype(np.float64)[::-1, :].copy()

spec = spec_from_v1_file(PARAMS)
res = autocalibrate_pv(
    v1, image, spec=spec,
    n_iter=4, half_window_px=8.0, snr_min=8.0,
    trim_mode='stratified_multfactor', trim_residual_pct=5.0,
    reuse_fits=True, lm_max_iter=300, verbose=False,
    distribution_report=False,
)
print(f'pipeline: strain {res.history[-1].mean_strain_uE:.2f} µε')


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  return torch.sparse_csr_tensor(indptr, indices, values,
/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(indptr, in

pipeline: strain 18.48 µε


In [2]:
fits = res.fits_final
def res_fn(unp):
    return pseudo_strain_residual(
        fits.Y_pix, fits.Z_pix, fits.ring_two_theta_deg, unp,
        rho_d=fits.rho_d, weights=fits.weights,
        ring_idx=fits.ring_idx, ring_d_spacing_A=fits.ring_d_spacing_A,
    )
with torch.no_grad():
    sigma_r = float((res_fn(res.unpacked) ** 2).mean() ** 0.5)
lap = fisher_at_map(spec, res_fn, res.unpacked,
                    sigma_r=sigma_r, ridge=1e-9,
                    dtype=torch.float64, device='cpu')

cov = lap.cov.detach().cpu().numpy()
def _flat(lap):
    out = []
    for n, o, s in zip(lap.refined_names, lap.refined_offsets, lap.refined_sizes):
        for k in range(s):
            out.append(f'{n}[{k}]' if s > 1 else n)
    return out
flat = _flat(lap)
name_to_idx = {n: i for i, n in enumerate(flat)}
print(f'Laplace cov ready: {len(flat)} refined dims')


Laplace cov ready: 20 refined dims


## Per-ring σ(Q) via the Jacobian chain rule

For each ring k:

$$Q_k = \frac{4\pi}{\lambda} \sin(\theta_k)$$

with θ_k determined by the observed ring radius R_k and L_sd.
The non-zero partials at the headline config (only L_sd refined
among Q-affecting parameters):

$$\frac{\partial Q_k}{\partial L_\mathrm{sd}} = \frac{4\pi}{\lambda} \cos(\theta_k) \cdot \frac{1}{2} \cdot \frac{-R_k \, p_x}{L_\mathrm{sd}^2 + (R_k p_x)^2}$$


In [3]:
rt = fits.rt
two_theta_deg = np.array(rt.two_theta_deg)
lam_A = float(res.unpacked['Wavelength'])
Lsd_um = float(res.unpacked['Lsd'])
pxY_um = float(res.unpacked['pxY'])

print(f'{"ring":>5s}  {"2θ°":>6s}  {"Q (Å⁻¹)":>10s}  {"d (Å)":>8s}  '
      f'{"σ(Q) [Å⁻¹]":>14s}  {"σ(Q)/Q [ppm]":>14s}')
for k, tt in enumerate(two_theta_deg):
    if tt <= 0:
        continue
    tt_rad = math.radians(tt); th_rad = 0.5 * tt_rad
    R_obs = Lsd_um * math.tan(tt_rad) / pxY_um
    Q = (4.0 * math.pi / lam_A) * math.sin(th_rad)
    d = 2.0 * math.pi / Q
    dtt_dLsd = -R_obs * pxY_um / (Lsd_um**2 + (R_obs * pxY_um)**2)
    dQ_dLsd = (4.0 * math.pi / lam_A) * math.cos(th_rad) * 0.5 * dtt_dLsd
    J_Q = np.zeros(len(flat))
    if 'Lsd' in name_to_idx:
        J_Q[name_to_idx['Lsd']] = dQ_dLsd
    var_Q = float(J_Q @ cov @ J_Q)
    sigma_Q = math.sqrt(max(var_Q, 0.0))
    print(f'  {k:>3d}  {tt:>6.2f}  {Q:>10.3f}  {d:>8.3f}  '
          f'{sigma_Q:>14.3e}  {sigma_Q/Q*1e6:>14.2f}')


 ring     2θ°     Q (Å⁻¹)     d (Å)      σ(Q) [Å⁻¹]    σ(Q)/Q [ppm]
    0    3.61       2.011     3.124       3.595e-06            1.79
    1    4.17       2.322     2.706       4.148e-06            1.79
    2    5.90       3.284     1.913       5.842e-06            1.78
    3    6.91       3.851     1.632       6.830e-06            1.77
    4    7.22       4.022     1.562       7.127e-06            1.77
    5    8.34       4.644     1.353       8.196e-06            1.76
    6    9.09       5.061     1.242       8.905e-06            1.76
    7    9.33       5.192     1.210       9.127e-06            1.76
    8   10.22       5.688     1.105       9.958e-06            1.75
    9   10.84       6.033     1.041       1.053e-05            1.75
   10   10.84       6.033     1.041       1.053e-05            1.75
   11   11.81       6.568     0.957       1.141e-05            1.74
   12   12.35       6.869     0.915       1.189e-05            1.73
   13   12.53       6.966     0.902       1.205e

## Putting σ(Q) into context

| Source of Q uncertainty | Typical magnitude |
|---|---|
| Calibration (this notebook) | ~10⁻⁶ Å⁻¹ |
| PDF bin width (`ΔQ`) | 10⁻³ Å⁻¹ |
| Counting noise on a peak | 10⁻³–10⁻² Å⁻¹ |
| Sample-displacement (capillary) | 10⁻³ Å⁻¹ |

**Calibration σ(Q) is 3 orders of magnitude below the bin width**
on a typical PDF reduction.  Unless you're doing absolute lattice
constant determination at sub-ppm precision, the calibration σ
isn't the limit.

## When calibration σ does matter

- **Multi-distance lattice-constant determination** (notebook 07):
  σ(a) at 2.83 ppm needs the data Cramér-Rao to be `< 1` ppm to
  avoid being calibration-limited.
- **High-precision pair-distance determination in PDF analysis**:
  if your bin width is set by Q resolution (typically `ΔQ = 0.005` Å⁻¹
  on a synchrotron PDF beamline), the per-Q-bin σ matters.
- **Strain-tensor reproducibility from FF-HEDM**: per-grain strain
  uncertainty includes the calibration σ contribution
  (see §10 in the paper).

## See also

- [`dev/paper/runners/run_sigma_q_propagation.py`](../dev/paper/runners/run_sigma_q_propagation.py) — full per-ring σ(Q) sweep
- §"Propagation of geometry σ to per-ring σ(Q)" in the paper
- [`tab:sigma_q`](../dev/paper/newPaper/main.tex) in the paper
